In [ ]:
#!/usr/bin/env python3
# xhs_province_pipeline.py


import os
import re
import time
import uuid
import random
import argparse
from datetime import datetime
from urllib.parse import quote
from typing import Optional, Dict, List

import pandas as pd
import openpyxl
from tqdm import tqdm

# ---------- 可选的浏览器驱动 ----------
try:
    from DrissionPage import ChromiumPage
except Exception as e:
    ChromiumPage = None  # 若未安装，抓取功能将不可用
    _drission_err = e

# ---------------- 配置（把省份与城市放在一起，方便修改） ----------------
from typing import Dict, List

DEFAULT_PROVINCE =  "北京","上海","天津","重庆"


from typing import Dict, List

DEFAULT_PROVINCE = "内蒙古"

PROVINCE_CITY_MAP: Dict[str, List[str]] = {
    # 直辖市
    "北京": ["北京"],
    "天津": ["天津"],
    "上海": ["上海"],
    "重庆": ["重庆"],
    
    # 华北地区
    "河北": [
        "石家庄", "唐山", "秦皇岛", "邯郸", "邢台", "保定", "张家口", "承德", "沧州", "廊坊", "衡水"
    ],
    "山西": [
        "太原", "大同", "阳泉", "长治", "晋城", "朔州", "晋中", "运城", "忻州", "临汾", "吕梁"
    ],
    "内蒙古": [
        "呼和浩特", "包头", "乌海", "赤峰", "通辽", "鄂尔多斯", "呼伦贝尔", "巴彦淖尔", "乌兰察布", "兴安盟", "锡林郭勒盟", "阿拉善盟"
    ],
    
    # 东北地区
    "辽宁": [
        "沈阳", "大连", "鞍山", "抚顺", "本溪", "丹东", "锦州", "营口", "阜新", "辽阳", "盘锦", "铁岭", "朝阳", "葫芦岛"
    ],
    "吉林": [
        "长春", "吉林", "四平", "辽源", "通化", "白山", "松原", "白城", "延边朝鲜族自治州"
    ],
    "黑龙江": [
        "哈尔滨", "齐齐哈尔", "鸡西", "鹤岗", "双鸭山", "大庆", "伊春", "佳木斯", "七台河", "牡丹江", "黑河", "绥化", "大兴安岭地区"
    ],
    
    # 华东地区
    "江苏": [
        "南京", "无锡", "徐州", "常州", "苏州", "南通", "连云港", "淮安", "盐城", "扬州", "镇江", "泰州", "宿迁"
    ],
    "浙江": [
        "杭州", "宁波", "温州", "嘉兴", "湖州", "绍兴", "金华", "衢州", "舟山", "台州", "丽水"
    ],
    "安徽": [
        "合肥", "芜湖", "蚌埠", "淮南", "马鞍山", "淮北", "铜陵", "安庆", "黄山", "滁州", "阜阳", "宿州", "六安", "亳州", "池州", "宣城"
    ],
    "福建": [
        "福州", "厦门", "莆田", "三明", "泉州", "漳州", "南平", "龙岩", "宁德"
    ],
    "江西": [
        "南昌", "景德镇", "萍乡", "九江", "新余", "鹰潭", "赣州", "吉安", "宜春", "抚州", "上饶"
    ],
    "山东": [
        "济南", "青岛", "淄博", "枣庄", "东营", "烟台", "潍坊", "济宁", "泰安", "威海", "日照", "临沂", "德州", "聊城", "滨州", "菏泽"
    ],
    
    # 华中地区
    "河南": [
        "郑州", "开封", "洛阳", "平顶山", "安阳", "鹤壁", "新乡", "焦作", "濮阳", "许昌", "漯河", "三门峡", "南阳", "商丘", "信阳", "周口", "驻马店", "济源"
    ],
    "湖北": [
        "武汉", "黄石", "十堰", "宜昌", "襄阳", "鄂州", "荆门", "孝感", "荆州", "黄冈", "咸宁", "随州", "恩施土家族苗族自治州", "仙桃", "潜江", "天门", "神农架"
    ],
    "湖南": [
        "长沙", "株洲", "湘潭", "衡阳", "邵阳", "岳阳", "常德", "张家界", "益阳", "郴州", "永州", "怀化", "娄底", "湘西土家族苗族自治州"
    ],
    
    # 华南地区
    "广东": [
        "广州", "韶关", "深圳", "珠海", "汕头", "佛山", "江门", "湛江", "茂名", "肇庆", "惠州", "梅州", "汕尾", "河源", "阳江", "清远", "东莞", "中山", "潮州", "揭阳", "云浮"
    ],
    "广西": [
        "南宁", "柳州", "桂林", "梧州", "北海", "防城港", "钦州", "贵港", "玉林", "百色", "贺州", "河池", "来宾", "崇左"
    ],
    "海南": [
        "海口", "三亚", "三沙", "儋州", "五指山", "琼海", "文昌", "万宁", "东方", "定安", "屯昌", "澄迈", "临高", "白沙", "昌江", "乐东", "陵水", "保亭", "琼中"
    ],
    
    # 西南地区
    "四川": [
        "成都", "自贡", "攀枝花", "泸州", "德阳", "绵阳", "广元", "遂宁", "内江", "乐山", "南充", "眉山", "宜宾", "广安", "达州", "雅安", "巴中", "资阳", "阿坝藏族羌族自治州", "甘孜藏族自治州", "凉山彝族自治州"
    ],
    "贵州": [
        "贵阳", "六盘水", "遵义", "安顺", "毕节", "铜仁", "黔西南布依族苗族自治州", "黔东南苗族侗族自治州", "黔南布依族苗族自治州"
    ],
    "云南": [
        "昆明", "曲靖", "玉溪", "保山", "昭通", "丽江", "普洱", "临沧", "楚雄彝族自治州", "红河哈尼族彝族自治州", "文山壮族苗族自治州", "西双版纳傣族自治州", "大理白族自治州", "德宏傣族景颇族自治州", "怒江傈僳族自治州", "迪庆藏族自治州"
    ],
    "西藏": [
        "拉萨", "日喀则", "昌都", "林芝", "山南", "那曲", "阿里地区"
    ],
    
    # 西北地区
    "陕西": [
        "西安", "铜川", "宝鸡", "咸阳", "渭南", "延安", "汉中", "榆林", "安康", "商洛"
    ],
    "甘肃": [
        "兰州", "嘉峪关", "金昌", "白银", "天水", "武威", "张掖", "平凉", "酒泉", "庆阳", "定西", "陇南", "临夏回族自治州", "甘南藏族自治州"
    ],
    "青海": [
        "西宁", "海东", "海北藏族自治州", "黄南藏族自治州", "海南藏族自治州", "果洛藏族自治州", "玉树藏族自治州", "海西蒙古族藏族自治州"
    ],
    "宁夏": [
        "银川", "石嘴山", "吴忠", "固原", "中卫"
    ],
    "新疆": [
        "乌鲁木齐", "克拉玛依", "吐鲁番", "哈密", "昌吉回族自治州", "博尔塔拉蒙古自治州", "巴音郭楞蒙古自治州", "阿克苏地区", "克孜勒苏柯尔克孜自治州", "喀什地区", "和田地区", "伊犁哈萨克自治州", "塔城地区", "阿勒泰地区", "石河子", "阿拉尔", "图木舒克", "五家渠", "铁门关"
    ]
}
# 若需要在脚本中默认覆盖城市列表，可修改下面 DEFAULT_CITIES（一般不需要）
CITIES: List[str] = PROVINCE_CITY_MAP.get(DEFAULT_PROVINCE, [])

YEAR_PREFIX = "2026"   # e.g. "2025年 " 或 ""
HOLIDAYS = ["元旦", "春节"]
EVENT = "无人机灯光秀"

# 抓取参数（可在 run 时覆盖）
MAX_SCROLLS_PER_QUERY = 5
SCROLL_PAUSE = (0.3, 0.8)
WAIT_AFTER_SEARCH = (0.2, 0.6)

# 页面选择器（可能需要按页面结构调整）
NOTE_SELECTOR = '.note-item'
TITLE_SELECTORS = ['.title', '.desc']
AUTHOR_SELECTOR = '.author'
CONTENT_SELECTOR = '.content'

# 关键词/权重（打分）
KEYWORD_REGEX = re.compile(r'无人机|灯光秀|灯光|无人机表演|无人机秀|表演|秀', re.I)
FIREWORK_REGEX = re.compile(r'烟花|焰火', re.I)
HOLIDAY_REGEX = re.compile(r'元旦|春节|除夕|元宵|大年初|跨年|正月', re.I)
TIME_KEYWORDS = ['2026']
MEDIA_KEYWORDS = ['新华社','人民网','央视','央视网','澎湃','日报','卫视','市政府','县政府','旅游局','文旅局']

WEIGHTS = {
    'time_keyword': 3,
    'title_keyword': 3,
    'content_keyword': 2,
    'holiday_word': 3,
    'city_in_text': 1,
    'media_author': 2,
    'firework': 1
}

UNRELATED_KEYWORDS = [
    '招聘','美食','酒店','房产','二手','求职','折扣','代购','转让','点评','攻略','租房','买房','团购'
]
UNRELATED_PENALTY_PER_KEY = 1
UNRELATED_PENALTY_MAX = 2
YEARS_PENALTY_REGEX = re.compile(r'2024|2025')

# ---------- 辅助函数（抓取与处理共用） ----------
def safe_get_text(elem, selector):
    try:
        el = elem.ele(selector, timeout=0)
        if not el:
            return ""
        return (el.text or "").strip()
    except:
        return ""

def safe_link(elem):
    try:
        a = elem.ele('tag:a', timeout=0)
        if a:
            return getattr(a, 'link', '') or ''
    except:
        return ""

def extract_key_info_from_text(text, radius=60):
    if not text:
        return ""
    s = text.replace('\n', ' ').strip()
    m = KEYWORD_REGEX.search(s)
    if not m:
        return s[:120]
    start = max(0, m.start() - radius)
    end = min(len(s), m.end() + radius)
    return s[start:end].strip()

def auto_resize_column(excel_path):
    try:
        wb = openpyxl.load_workbook(excel_path)
        ws = wb.active
        for col in ws.columns:
            max_len = max(len(str(cell.value or "")) for cell in col)
            ws.column_dimensions[col[0].column_letter].width = min(80, max_len + 2)
        wb.save(excel_path)
    except Exception:
        pass

# 打分函数（含惩罚）
def score_record(title, content, author, city, holiday_token, city_from_query=False):
    s = 0
    txt = " ".join([title or "", content or ""])

    # 正向加分
    for tk in TIME_KEYWORDS:
        if tk and tk in txt:
            s += WEIGHTS.get('time_keyword', 0)
            break
    if title and KEYWORD_REGEX.search(title):
        s += WEIGHTS.get('title_keyword', 0)
    if content and KEYWORD_REGEX.search(content):
        s += WEIGHTS.get('content_keyword', 0)
    if HOLIDAY_REGEX.search(txt) or (holiday_token and holiday_token in txt):
        s += WEIGHTS.get('holiday_word', 0)
    # 城市出现且未同时出现其他城市名
    if city:
        other_city_found = False
        for oc in CITIES:
            if oc == city:
                continue
            if oc in txt:
                other_city_found = True
                break
        if (city in txt) and not other_city_found:
            s += WEIGHTS.get('city_in_text', 0)
    if FIREWORK_REGEX.search(txt):
        s += WEIGHTS.get('firework', 0)
    if author:
        low = author.lower()
        for mk in MEDIA_KEYWORDS:
            if mk.lower() in low:
                s += WEIGHTS.get('media_author', 0)
                break

    # 惩罚
    if YEARS_PENALTY_REGEX.search(txt):
        s -= 1
    if city_from_query and city:
        if city not in txt:
            s -= 2
    if txt:
        cnt_unrelated = sum(1 for kw in UNRELATED_KEYWORDS if kw in txt)
        if cnt_unrelated > 0:
            penalty = min(cnt_unrelated * UNRELATED_PENALTY_PER_KEY, UNRELATED_PENALTY_MAX)
            s -= penalty
    return s

def possibility_from_max(mx):
    if mx >= 10:
        return '极度可能'
    if mx == 9:
        return '高度可能'
    if mx == 8:
        return '中度可能'
    return '低可能'

def activity_from_mean(mean):
    if mean >= 5.0:
        return '活跃'
    if mean >= 3.0:
        return '一般活跃'
    return '不活跃'

def apply_post_weighting(summary_full,
                         lam=0.6,    # 权重灵敏度
                         w_min=0.5,
                         w_max=1.8,
                         active_thr=3.0,
                         mid_thr=2.0):
    df = summary_full.copy()
    mean_posts = df['n_records'].mean() if 'n_records' in df.columns else 0.0
    if not mean_posts or mean_posts <= 0:
        mean_posts = 1.0
    df['post_ratio'] = df['n_records'] / mean_posts
    df['weight'] = (1.0 + lam * (df['post_ratio'] - 1.0)).clip(lower=w_min, upper=w_max)
    df['weighted_activity_score'] = df['score_mean'] * df['weight']
    def _level(s):
        if s >= active_thr:
            return '活跃'
        if s >= mid_thr:
            return '一般活跃'
        return '不活跃'
    df['activity_level_weighted'] = df['weighted_activity_score'].apply(_level)
    return df

# ---------- 抓取主流程 ----------
def run_presence_detection(output_raw: str,
                           output_summary_long: str,
                           output_summary_wide: str,
                           queries: Optional[List[str]] = None,
                           note_selector=NOTE_SELECTOR,
                           max_scrolls_per_query=MAX_SCROLLS_PER_QUERY,
                           scroll_pause=SCROLL_PAUSE,
                           wait_after_search=WAIT_AFTER_SEARCH,
                           stop_after_n_queries: Optional[int]=None):
    """
    使用 DrissionPage 抓取小红书（note 列表），对每条记录打分并保存 raw 与汇总。
    返回 (raw_df, summary_full_df, wide_df)
    """
    if ChromiumPage is None:
        raise RuntimeError(f"DrissionPage 未安装或不可用，无法抓取。错误：{_drission_err}")

    if queries is None:
        # fallback: generate queries from current global CITIES
        queries_local = []
        for h in HOLIDAYS:
            token = (YEAR_PREFIX + h).strip()
            for c in CITIES:
                queries_local.append(f"{token} {c} {EVENT}")
                queries_local.append(f"{c} {token} {EVENT}")
        queries = list(dict.fromkeys(queries_local))

    if stop_after_n_queries:
        queries = list(queries)[:stop_after_n_queries]

    page = ChromiumPage()
    print("打开 Chromium，请在弹出的窗口中扫码登录（如需要）...")
    try:
        page.get('https://www.xiaohongshu.com/explore')
    except:
        pass
    time.sleep(1)

    records = []
    with tqdm(queries, desc="Queries", unit="q") as tq:
        for q in tq:
            time.sleep(random.uniform(*wait_after_search))
            search_url = "https://www.xiaohongshu.com/search_result?keyword=" + quote(q)
            try:
                page.get(search_url)
            except:
                try:
                    page.execute_script("window.location.href = '%s'" % search_url)
                except:
                    pass
            time.sleep(random.uniform(*wait_after_search))

            # 识别 query 中是否包含某城市（用于 city_from_query）
            city_in_query = ""
            for c in CITIES:
                if c in q:
                    city_in_query = c
                    break

            for _ in range(max_scrolls_per_query):
                try:
                    cards = page.eles(note_selector) or []
                except:
                    cards = []

                for card in cards:
                    try:
                        url = safe_link(card) or ""
                        title = ""
                        for sel in TITLE_SELECTORS:
                            title = safe_get_text(card, sel)
                            if title:
                                break
                        author = safe_get_text(card, AUTHOR_SELECTOR)
                        content = safe_get_text(card, CONTENT_SELECTOR) or (card.text or "")
                        key_info = extract_key_info_from_text(" ".join([title, content, q]))

                        # 城市与节日从 query 推断（优先）
                        city = ""
                        holiday_token = ""
                        city_from_query = False

                        if city_in_query:
                            city = city_in_query
                            city_from_query = True

                        # query 中找节日
                        for h in HOLIDAYS:
                            if h in q or (YEAR_PREFIX and (YEAR_PREFIX.strip() + h) in q):
                                holiday_token = (YEAR_PREFIX + h).strip()
                                break

                        # 如果 query 未提供 city，则从文本中识别（优先第一个匹配城市）
                        if not city:
                            txt_all = " ".join([title or "", content or ""])
                            for c in CITIES:
                                if c in txt_all:
                                    city = c
                                    city_from_query = False
                                    break

                        # 如果 query 未给节日，再尝试从文本识别
                        if not holiday_token:
                            txt_all = " ".join([title or "", content or ""])
                            for h in HOLIDAYS:
                                if h in txt_all:
                                    holiday_token = h
                                    break

                        score = score_record(title, content, author, city, holiday_token, city_from_query=city_from_query)

                        rec = {
                            "query": q,
                            "holiday": holiday_token,
                            "city": city,
                            "title": title,
                            "author": author,
                            "url": url,
                            "content": (content or "")[:500],
                            "key_info": key_info,
                            "score": score
                        }
                        records.append(rec)
                    except Exception:
                        continue

                try:
                    page.scroll.down(800)
                except:
                    try:
                        page.execute_script("window.scrollBy(0, 800);")
                    except:
                        pass
                time.sleep(random.uniform(*scroll_pause))

    if not records:
        print("未抓到记录，检查登录或选择器是否正确。")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    df = pd.DataFrame(records)
    df.drop_duplicates(subset=["url","title"], inplace=True)
    df.fillna("", inplace=True)

    # 保存原始记录
    try:
        df.to_excel(output_raw, index=False)
        auto_resize_column(output_raw)
        print(f"已保存原始记录（含得分）到 {output_raw}")
    except Exception as e:
        print("保存原始记录失败：", e)

    # 聚合统计（city x holiday）
    grp = df.groupby(['city','holiday'])
    stats = grp['score'].agg(['max','mean','count']).reset_index().rename(columns={
        'max':'score_max','mean':'score_mean','count':'n_records'
    })

    # 代表性记录：score == score_max 的记录，按 score 降序取首条
    reps = df.merge(stats[['city','holiday','score_max']],
                    left_on=['city','holiday','score'],
                    right_on=['city','holiday','score_max'],
                    how='inner')

    if not reps.empty:
        reps = reps.sort_values(['city','holiday','score'], ascending=[True, True, False]) \
                   .drop_duplicates(subset=['city','holiday'], keep='first') \
                   .loc[:, ['city', 'holiday', 'title', 'author', 'url', 'key_info']].reset_index(drop=True)
        reps.columns = ['city','holiday','title_rep','author_rep','url_rep','key_info_rep']
    else:
        reps = pd.DataFrame(columns=['city','holiday','title_rep','author_rep','url_rep','key_info_rep'])

    summary = stats.merge(reps, on=['city','holiday'], how='left')

    # 确保每个 city×holiday 都存在（补零行）
    rows = []
    for c in CITIES:
        for h in HOLIDAYS:
            token = (YEAR_PREFIX + h).strip()
            r = summary[(summary['city']==c) & (summary['holiday']==token)]
            if r.empty:
                rows.append({
                    'city': c,
                    'holiday': token,
                    'score_max': 0,
                    'score_mean': 0.0,
                    'n_records': 0,
                    'title_rep': '',
                    'author_rep': '',
                    'url_rep': '',
                    'key_info_rep': ''
                })
            else:
                rows.append({
                    'city': c,
                    'holiday': token,
                    'score_max': int(r['score_max'].iloc[0]),
                    'score_mean': float(r['score_mean'].iloc[0]),
                    'n_records': int(r['n_records'].iloc[0]),
                    'title_rep': r['title_rep'].iloc[0] or '',
                    'author_rep': r['author_rep'].iloc[0] or '',
                    'url_rep': r['url_rep'].iloc[0] or '',
                    'key_info_rep': r['key_info_rep'].iloc[0] or ''
                })
    summary_full = pd.DataFrame(rows)

    # --- 加权活跃度（将帖子数纳入权重） ---
    summary_full = apply_post_weighting(summary_full,
                                        lam=0.6, w_min=0.5, w_max=1.8,
                                        active_thr=3.0, mid_thr=2.0)

    # 保留未加权 activity_level 以便比较
    summary_full['activity_level_unweighted'] = summary_full['score_mean'].apply(activity_from_mean)

    # label: score_max >= 9 -> label = 1（改为 9 分阈值）
    summary_full['label'] = summary_full['score_max'].apply(lambda x: 1 if x >= 9 else 0)

    # possibility（基于 score_max）
    summary_full['possibility'] = summary_full['score_max'].apply(possibility_from_max)

    # 不确定性标注，供人工复核
    def uncertainty_flag(row):
        mx = row['score_max']
        mean = row['score_mean']
        if row['n_records'] == 0:
            return 'low'
        if (9 <= mx <= 10) or (4.5 <= mean <= 5.5):
            return 'uncertain'
        if mx >= 10 or mean >= 6.0:
            return 'high'
        return 'low'
    summary_full['uncertainty'] = summary_full.apply(uncertainty_flag, axis=1)

    # 最终列顺序
    summary_full = summary_full[['city','holiday','n_records','post_ratio','weight',
                                 'score_max','score_mean','activity_level_unweighted',
                                 'weighted_activity_score','activity_level_weighted',
                                 'label','possibility','uncertainty',
                                 'title_rep','author_rep','url_rep','key_info_rep']]

    # 保存长表
    try:
        summary_full.to_excel(output_summary_long, index=False)
        auto_resize_column(output_summary_long)
        print(f"已保存长表汇总到 {output_summary_long}")
    except Exception as e:
        print("保存长表失败：", e)

    # 宽表（每 city 一行）
    wide = summary_full.pivot(index='city', columns='holiday',
                              values=['n_records','post_ratio','weight','score_max','score_mean',
                                      'activity_level_unweighted','weighted_activity_score','activity_level_weighted',
                                      'label','possibility','uncertainty'])
    wide.columns = [f"{col[1]}_{col[0]}" for col in wide.columns]
    wide = wide.reset_index().fillna(0)

    try:
        wide.to_excel(output_summary_wide, index=False)
        auto_resize_column(output_summary_wide)
        print(f"已保存宽表汇总到 {output_summary_wide}")
    except Exception as e:
        print("保存宽表失败：", e)

    return df, summary_full, wide

# ---------- 从 raw 生成简化宽表并合并 GDP ----------
def generate_wide_minimal_with_gdp(raw_excel_path: str,
                                   output_path: str = "xhs_province_presence_summary_wide_minimal.xlsx",
                                   recompute_scores: bool = False,
                                   activity_use_weighted: bool = True,
                                   gdp_csv_path: Optional[str] = None,
                                   gdp_mapping: Optional[Dict[str, float]] = None,
                                   gdp_sheet: Optional[str] = None,
                                   sort_by_gdp: bool = False,
                                   province: str = "province"):
    """
    生成最终简化宽表并可合并 GDP。
    返回：wide DataFrame
    """
    # 读取 raw
    df = pd.read_excel(raw_excel_path)
    df = df.fillna("")

    # 如果需要重算 score 或缺失 score 列
    if recompute_scores or ('score' not in df.columns):
        def _city_from_query_flag(row):
            q = str(row.get('query', '') or '')
            city = str(row.get('city', '') or '')
            if not city:
                return False
            return city in q
        scores = []
        for _, row in df.iterrows():
            title = str(row.get('title', '') or '')
            content = str(row.get('content', '') or '')
            author = str(row.get('author', '') or '')
            city = str(row.get('city', '') or '')
            holiday_token = str(row.get('holiday', '') or '')
            city_from_query = _city_from_query_flag(row)
            sc = score_record(title, content, author, city, holiday_token, city_from_query=city_from_query)
            scores.append(sc)
        df['score'] = scores

    # 去重（与原脚本行为一致）
    if 'url' in df.columns and 'title' in df.columns:
        df = df.drop_duplicates(subset=['url','title'])
    df = df.reset_index(drop=True)

    # 按 city x holiday 聚合
    grp = df.groupby(['city','holiday'])
    stats = grp['score'].agg(['max','mean','count']).reset_index().rename(columns={
        'max':'score_max','mean':'score_mean','count':'n_records'
    })

    # 补全所有 city x holiday
    rows = []
    for c in CITIES:
        for h in HOLIDAYS:
            token = (YEAR_PREFIX + h).strip()
            r = stats[(stats['city']==c) & (stats['holiday']==token)]
            if r.empty:
                rows.append({'city': c, 'holiday': token, 'score_max': 0, 'score_mean': 0.0, 'n_records': 0})
            else:
                rows.append({
                    'city': c,
                    'holiday': token,
                    'score_max': int(r['score_max'].iloc[0]),
                    'score_mean': float(r['score_mean'].iloc[0]),
                    'n_records': int(r['n_records'].iloc[0])
                })
    summary_full = pd.DataFrame(rows)

    # label 与未加权 activity（将 label 阈值改为 9）
    summary_full['label'] = summary_full['score_max'].apply(lambda x: 1 if x >= 9 else 0)
    summary_full['activity_unweighted'] = summary_full['score_mean'].apply(activity_from_mean)

    # 加权 activity（用以合并）
    weighted = apply_post_weighting(summary_full[['city','holiday','n_records','score_max','score_mean']].copy())
    summary_full = summary_full.merge(weighted[['city','holiday','activity_level_weighted']],
                                      on=['city','holiday'], how='left')

    # 选择 activity 列
    summary_full['activity'] = summary_full['activity_level_weighted'] if activity_use_weighted else summary_full['activity_unweighted']

    # 仅保留需要列并 pivot 成宽表
    keep_cols = ['city','holiday','n_records','score_max','score_mean','label','activity']
    small = summary_full[keep_cols]

    # Pivot：每个 holiday 变成一组列，列名格式：{holiday}_{metric}
    wide = small.pivot(index='city', columns='holiday',
                       values=['n_records','score_max','score_mean','label','activity'])
    # 规范列名：{holiday}_{metric}
    wide.columns = [f"{col[1]}_{col[0]}" for col in wide.columns]
    wide = wide.reset_index()

    # 为每个节日添加 possibility（基于对应的 score_max）
    for h in HOLIDAYS:
        score_max_col = f"{h}_score_max"
        poss_col = f"{h}_possibility"
        if score_max_col in wide.columns:
            wide[poss_col] = wide[score_max_col].apply(lambda x: possibility_from_max(int(x)) if pd.notna(x) else '低可能')
        else:
            wide[poss_col] = '低可能'

    # 计算元旦与春节的 score_mean 的平均值
    score_mean_cols = [f"{h}_score_mean" for h in HOLIDAYS]
    for col in score_mean_cols:
        if col not in wide.columns:
            wide[col] = 0.0
    wide['avg_score_mean'] = wide[score_mean_cols].astype(float).mean(axis=1)

    # 确保活动列的缺失填为 '不活跃'
    for h in HOLIDAYS:
        col = f"{h}_activity"
        if col in wide.columns:
            wide[col] = wide[col].fillna('不活跃')

    # ---------------- 合并 GDP（可选） ----------------
    gdp_df = None
    if gdp_csv_path:
        try:
            if str(gdp_csv_path).lower().endswith(('.xls','.xlsx')):
                gdp_df = pd.read_excel(gdp_csv_path, sheet_name=gdp_sheet)
            else:
                gdp_df = pd.read_csv(gdp_csv_path)
        except Exception as e:
            print("[warning] 读取 GDP 文件失败：", e)
            gdp_df = None

    if (gdp_df is None) and (gdp_mapping is not None):
        try:
            gdp_df = pd.DataFrame([{'city': k, 'gdp': float(v)} for k,v in gdp_mapping.items()])
        except Exception as e:
            print("[warning] 构建 GDP DataFrame 失败：", e)
            gdp_df = None

    if isinstance(gdp_df, pd.DataFrame):
        # 识别 city/gdp 列
        city_col = None
        gdp_col = None
        for c in gdp_df.columns:
            if c.lower() in ('city','城市','name'):
                city_col = c
            if c.lower() in ('gdp','value','amount','数值','gdp_value'):
                gdp_col = c
        if city_col is None:
            city_col = gdp_df.columns[0]
        if gdp_col is None and len(gdp_df.columns) > 1:
            gdp_col = gdp_df.columns[1]
        if gdp_col is None:
            print("[warning] 未找到 GDP 列，跳过 GDP 合并。")
        else:
            gdp_df = gdp_df[[city_col, gdp_col]].rename(columns={city_col:'city', gdp_col:'gdp'})
            gdp_df['gdp'] = pd.to_numeric(gdp_df['gdp'].astype(str).str.replace(r'[^\d\.\-]', '', regex=True), errors='coerce')
            wide = wide.merge(gdp_df, on='city', how='left')
            wide['gdp_rank'] = wide['gdp'].rank(method='dense', ascending=False)
            wide['gdp_rank'] = wide['gdp_rank'].where(~wide['gdp_rank'].isna(), None)
            try:
                wide['gdp_rank'] = wide['gdp_rank'].astype('Int64')
            except Exception:
                pass
            if sort_by_gdp:
                wide = wide.sort_values(by=['gdp'], ascending=False).reset_index(drop=True)

    # 保存
    try:
        wide.to_excel(output_path, index=False)
        auto_resize_column(output_path)
        print(f"[saved] {output_path}")
    except Exception as e:
        print("保存失败：", e)

    return wide

# ---------- CLI / 主入口 ----------
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="一体化：抓取小红书 -> 保存 raw -> 生成宽表（可合并 GDP）。")
    parser.add_argument("--province", "-p", default=DEFAULT_PROVINCE, help="省份名称（用于文件命名与文件夹，且从 PROVINCE_CITY_MAP 中读取地级市列表）")
    parser.add_argument("--cities", default=None, help="可选：以逗号分隔覆盖默认地级市列表（例如 '南宁,柳州,桂林'）")
    parser.add_argument("--no_scrape", action="store_true", help="跳过抓取阶段（仅处理已有 raw 文件）")
    parser.add_argument("--raw", "-r", default=None, help="若跳过抓取，请指定 raw excel 路径")
    parser.add_argument("--stop_queries", type=int, default=None, help="调试时仅运行前 N 个查询（可选）")
    parser.add_argument("--max_scrolls", type=int, default=MAX_SCROLLS_PER_QUERY, help="每个 query 的最大下拉次数")
    parser.add_argument("--scroll_pause_min", type=float, default=SCROLL_PAUSE[0], help="scroll pause min")
    parser.add_argument("--scroll_pause_max", type=float, default=SCROLL_PAUSE[1], help="scroll pause max")
    parser.add_argument("--recompute_scores", action="store_true", help="在生成宽表阶段强制重算 score")
    parser.add_argument("--gdp_csv", default=None, help="可选：GDP 文件路径（CSV 或 XLSX）")
    parser.add_argument("--sort_by_gdp", action="store_true", help="若提供 GDP，则按 GDP 排序输出")
    args, unknown = parser.parse_known_args()

    # 确定城市列表（优先使用 --cities 覆盖，其次使用 PROVINCE_CITY_MAP）
    if args.cities:
        CITIES = [c.strip() for c in args.cities.split(",") if c.strip()]
    else:
        CITIES = PROVINCE_CITY_MAP.get(args.province, PROVINCE_CITY_MAP.get(DEFAULT_PROVINCE, []))

    if not CITIES:
        raise SystemExit(f"未找到 {args.province} 的城市列表，请在 PROVINCE_CITY_MAP 中添加或使用 --cities 参数覆盖。")

    # 根据最终 CITIES 生成 QUERIES（用于抓取）
    QUERIES = []
    for h in HOLIDAYS:
        token = (YEAR_PREFIX + h).strip()
        for c in CITIES:
            QUERIES.append(f"{token} {c} {EVENT}")
            QUERIES.append(f"{c} {token} {EVENT}")
    QUERIES = list(dict.fromkeys(QUERIES))

    # 创建输出文件夹（每次运行新建）
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    short = uuid.uuid4().hex[:6]
    safe_prov = str(args.province).replace(" ", "_")
    out_folder = f"{safe_prov}_xhs_{ts}_{short}"
    try:
        os.makedirs(out_folder, exist_ok=True)
    except Exception as e:
        print("[warning] 创建输出文件夹失败：", e)
        out_folder = "."

    # 组装文件路径（基于省份）
    raw_default = os.path.join(out_folder, f"xhs_{safe_prov}_presence_raw.xlsx")
    summary_long_default = os.path.join(out_folder, f"xhs_{safe_prov}_presence_summary_long.xlsx")
    summary_wide_default = os.path.join(out_folder, f"xhs_{safe_prov}_presence_summary_wide.xlsx")
    wide_minimal_default = os.path.join(out_folder, f"xhs_{safe_prov}_presence_summary_wide_minimal.xlsx")

    print(f"输出目录: {out_folder}")
    print(f"使用省份: {args.province}")
    print(f"城市列表 ({len(CITIES)}): {CITIES}")

    # 1) 抓取（除非 --no_scrape）
    raw_input_path = args.raw if args.raw else raw_default
    if args.no_scrape:
        if not args.raw:
            raise SystemExit("跳过抓取时必须通过 --raw 指定已有 raw 文件路径。")
        print("跳过抓取阶段，使用提供的 raw 文件：", args.raw)
    else:
        if ChromiumPage is None:
            raise SystemExit(f"DrissionPage 未安装或不可用，无法抓取。错误：{_drission_err}")
        print("开始抓取阶段，raw 将保存到：", raw_default)
        # 覆盖全局抓取参数并传入 QUERIES
        df_raw, summary_long, summary_wide = run_presence_detection(
            output_raw=raw_default,
            output_summary_long=summary_long_default,
            output_summary_wide=summary_wide_default,
            queries=QUERIES,
            note_selector=NOTE_SELECTOR,
            max_scrolls_per_query=args.max_scrolls,
            scroll_pause=(args.scroll_pause_min, args.scroll_pause_max),
            wait_after_search=WAIT_AFTER_SEARCH,
            stop_after_n_queries=args.stop_queries
        )
        raw_input_path = raw_default

    # 2) 从 raw 生成简化宽表并合并 GDP（若提供）
    print("开始从 raw 生成简化宽表，输出到：", wide_minimal_default)
    wide = generate_wide_minimal_with_gdp(
        raw_excel_path=raw_input_path,
        output_path=wide_minimal_default,
        recompute_scores=args.recompute_scores,
        activity_use_weighted=True,
        gdp_csv_path=args.gdp_csv,
        gdp_mapping=None,
        gdp_sheet=None,
        sort_by_gdp=args.sort_by_gdp,
        province=args.province
    )

    print("完成。输出文件夹：", out_folder)
    try:
        print("生成的宽表预览：")
        print(wide.head(20).to_string(index=False))
    except Exception:
        pass

输出目录: 内蒙古_xhs_20260306_072030_890938
使用省份: 内蒙古
城市列表 (12): ['呼和浩特', '包头', '乌海', '赤峰', '通辽', '鄂尔多斯', '呼伦贝尔', '巴彦淖尔', '乌兰察布', '兴安盟', '锡林郭勒盟', '阿拉善盟']
开始抓取阶段，raw 将保存到： 内蒙古_xhs_20260306_072030_890938\xhs_内蒙古_presence_raw.xlsx
打开 Chromium，请在弹出的窗口中扫码登录（如需要）...


Queries:   4%|▍         | 2/48 [01:51<49:15, 64.24s/q]

In [ ]:

#!/usr/bin/env python3
# xhs_national_pro_final.py

import os
import re
import time
import random
import argparse
from datetime import datetime
from urllib.parse import quote
from typing import Dict, List

import pandas as pd
import openpyxl
from tqdm import tqdm

# --- 1. 核心驱动加载 ---
try:
    from DrissionPage import ChromiumPage
except ImportError:
    print("请先安装驱动: pip install DrissionPage")
    ChromiumPage = None

# --- 2. 配置信息 ---
PROVINCE_CITY_MAP: Dict[str, List[str]] = {
    "北京": ["北京"], "天津": ["天津"], "上海": ["上海"], "重庆": ["重庆"],
    "河北": ["石家庄", "唐山", "秦皇岛", "邯郸", "邢台", "保定", "张家口", "承德", "沧州", "廊坊", "衡水"],
    "山西": ["太原", "大同", "阳泉", "长治", "晋城", "朔州", "晋中", "运城", "忻州", "临汾", "吕梁"],
    "内蒙古": ["呼和浩特", "包头", "乌海", "赤峰", "通辽", "鄂尔多斯", "呼伦贝尔", "巴彦淖尔", "乌兰察布", "兴安盟", "锡林郭勒盟", "阿拉善盟"],
    "辽宁": ["沈阳", "大连", "鞍山", "抚顺", "本溪", "丹东", "锦州", "营口", "阜新", "辽阳", "盘锦", "铁岭", "朝阳", "葫芦岛"],
    "吉林": ["长春", "吉林", "四平", "辽源", "通化", "白山", "松原", "白城", "延边朝鲜族自治州"],
    "黑龙江": ["哈尔滨", "齐齐哈尔", "鸡西", "鹤岗", "双鸭山", "大庆", "伊春", "佳木斯", "七台河", "牡丹江", "黑河", "绥化", "大兴安岭地区"],
    "江苏": ["南京", "无锡", "徐州", "常州", "苏州", "南通", "连云港", "淮安", "盐城", "扬州", "镇江", "泰州", "宿迁"],
    "浙江": ["杭州", "宁波", "温州", "嘉兴", "湖州", "绍兴", "金华", "衢州", "舟山", "台州", "丽水"],
    "安徽": ["合肥", "芜湖", "蚌埠", "淮南", "马鞍山", "淮北", "铜陵", "安庆", "黄山", "滁州", "阜阳", "宿州", "六安", "亳州", "池州", "宣城"],
    "福建": ["福州", "厦门", "莆田", "三明", "泉州", "漳州", "南平", "龙岩", "宁德"],
    "江西": ["南昌", "景德镇", "萍乡", "九江", "新余", "鹰潭", "赣州", "吉安", "宜春", "抚州", "上饶"],
    "山东": ["济南", "青岛", "淄博", "枣庄", "东营", "烟台", "潍坊", "济宁", "泰安", "威海", "日照", "临沂", "德州", "聊城", "滨州", "菏泽"],
    "河南": ["郑州", "开封", "洛阳", "平顶山", "安阳", "鹤壁", "新乡", "焦作", "濮阳", "许昌", "漯河", "三门峡", "南阳", "商丘", "信阳", "周口", "驻马店", "济源"],
    "湖北": ["武汉", "黄石", "十堰", "宜昌", "襄阳", "鄂州", "荆门", "孝感", "荆州", "黄冈", "咸宁", "随州", "恩施土家族苗族自治州", "仙桃", "潜江", "天门", "神农架"],
    "湖南": ["长沙", "株洲", "湘潭", "衡阳", "邵阳", "岳阳", "常德", "张家界", "益阳", "郴州", "永州", "怀化", "娄底", "湘西土家族苗族自治州"],
    "广东": ["广州", "韶关", "深圳", "珠海", "汕头", "佛山", "江门", "湛江", "茂名", "肇庆", "惠州", "梅州", "汕尾", "河源", "阳江", "清远", "东莞", "中山", "潮州", "揭阳", "云浮"],
    "广西": ["南宁", "柳州", "桂林", "梧州", "北海", "防城港", "钦州", "贵港", "玉林", "百色", "贺州", "河池", "来宾", "崇左"],
    "海南": ["海口", "三亚", "三沙", "儋州", "五指山", "琼海", "文昌", "万宁", "东方", "定安", "屯昌", "澄迈", "临高", "白沙", "昌江", "乐东", "陵水", "保亭", "琼中"],
    "四川": ["成都", "自贡", "攀枝花", "泸州", "德阳", "绵阳", "广元", "遂宁", "内江", "乐山", "南充", "眉山", "宜宾", "广安", "达州", "雅安", "巴中", "资阳", "阿坝藏族羌族自治州", "甘孜藏族自治州", "凉山彝族自治州"],
    "贵州": ["贵阳", "六盘水", "遵义", "安顺", "毕节", "铜仁", "黔西南布依族苗族自治州", "黔东南苗族侗族自治州", "黔南布依族苗族自治州"],
    "云南": ["昆明", "曲靖", "玉溪", "保山", "昭通", "丽江", "普洱", "临沧", "楚雄彝族自治州", "红河哈尼族彝族自治州", "文山壮族苗族自治州", "西双版纳傣族自治州", "大理白族自治州", "德宏傣族景颇族自治州", "怒江傈僳族自治州", "迪庆藏族自治州"],
    "西藏": ["拉萨", "日喀则", "昌都", "林芝", "山南", "那曲", "阿里地区"],
    "陕西": ["西安", "铜川", "宝鸡", "咸阳", "渭南", "延安", "汉中", "榆林", "安康", "商洛"],
    "甘肃": ["兰州", "嘉峪关", "金昌", "白银", "天水", "武威", "张掖", "平凉", "酒泉", "庆阳", "定西", "陇南", "临夏回族自治州", "甘南藏族自治州"],
    "青海": ["西宁", "海东", "海北藏族自治州", "黄南藏族自治州", "海南藏族自治州", "果洛藏族自治州", "玉树藏族自治州", "海西蒙古族藏族自治州"],
    "宁夏": ["银川", "石嘴山", "吴忠", "固原", "中卫"],
    "新疆": ["乌鲁木齐", "克拉玛依", "吐鲁番", "哈密", "昌吉回族自治州", "博尔塔拉蒙古自治州", "巴音郭楞蒙古自治州", "阿克苏地区", "克孜勒苏柯尔克孜自治州", "喀什地区", "和田地区", "伊犁哈萨克自治州", "塔城地区", "阿勒泰地区", "石河子", "阿拉尔", "图木舒克", "五家渠", "铁门关"]
}

MUNICIPALITIES = ["北京", "上海", "天津", "重庆"]
SCROLL_MUNICIPALITY = 15  # 直辖市深度
SCROLL_NORMAL = 5        # 普通城市深度

YEAR_PREFIX = "2026"
HOLIDAYS = ["元旦", "春节"]
EVENT_KW = "无人机灯光秀"

# --- 3. 打分逻辑权重 ---
KEYWORD_REGEX = re.compile(r'无人机|灯光秀|表演|秀', re.I)
MEDIA_KEYWORDS = ['新华社','人民网','央视','澎湃','日报','卫视','政府','文旅局']
WEIGHTS = {'time': 3, 'title': 3, 'holiday': 3, 'media': 2}

# --- 4. 辅助函数 ---
def safe_get_text(elem, selector):
    try:
        el = elem.ele(selector, timeout=0)
        return (el.text or "").strip() if el else ""
    except: return ""

def score_record(title, author, city, holiday):
    s = 0
    txt = title or ""
    if "2026" in txt: s += WEIGHTS['time']
    if KEYWORD_REGEX.search(txt): s += WEIGHTS['title']
    if holiday in txt: s += WEIGHTS['holiday']
    if author and any(mk in author for mk in MEDIA_KEYWORDS): s += WEIGHTS['media']
    return s

def auto_resize_column(excel_path):
    try:
        wb = openpyxl.load_workbook(excel_path)
        ws = wb.active
        for col in ws.columns:
            max_len = max(len(str(cell.value or "")) for cell in col)
            ws.column_dimensions[col[0].column_letter].width = min(50, max_len + 2)
        wb.save(excel_path)
    except: pass

# --- 5. 执行主流程 ---
def run_national_task(target_provinces):
    if ChromiumPage is None: return
    page = ChromiumPage()
    
    all_data = []
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_dir = f"xhs_result_{ts}"
    os.makedirs(out_dir, exist_ok=True)
    
    # 展平任务
    task_queue = []
    for prov in target_provinces:
        for city in PROVINCE_CITY_MAP[prov]:
            task_queue.append((prov, city))

    print(f"启动任务：共 {len(task_queue)} 个城市。直辖市深度：{SCROLL_MUNICIPALITY}")

    for prov, city in tqdm(task_queue, desc="全城监测中"):
        # 判断抓取深度
        scroll_times = SCROLL_MUNICIPALITY if city in MUNICIPALITIES else SCROLL_NORMAL
        
        for h in HOLIDAYS:
            query = f"{YEAR_PREFIX} {city} {h} {EVENT_KW}"
            try:
                page.get(f"https://www.xiaohongshu.com/search_result?keyword={quote(query)}")
                time.sleep(random.uniform(1.0, 1.5)) # 基础等待
                
                for _ in range(scroll_times):
                    cards = page.eles('.note-item')
                    for card in cards:
                        try:
                            title = next((safe_get_text(card, s) for s in ['.title', '.desc'] if safe_get_text(card, s)), "")
                            author = safe_get_text(card, '.author')
                            url = card.ele('tag:a').link
                            score = score_record(title, author, city, h)
                            
                            all_data.append({
                                "province": prov, "city": city, "holiday": h,
                                "title": title, "author": author, "url": url, "score": score
                            })
                        except: continue
                    page.scroll.down(1000)
                    time.sleep(random.uniform(0.5, 0.8))
            except Exception as e:
                print(f"跳过 {city}-{h}: {e}")
                continue

    if not all_data:
        print("未采集到数据。")
        return

    # --- 6. 数据后处理与汇总 ---
    df_raw = pd.DataFrame(all_data).drop_duplicates(subset=['url', 'title'])
    raw_path = os.path.join(out_dir, "national_raw.xlsx")
    df_raw.to_excel(raw_path, index=False)

    # 聚合：计数、平均分、最高分
    stats = df_raw.groupby(['province', 'city', 'holiday'])['score'].agg(
        n_records='count',
        score_mean='mean',
        score_max='max'
    ).reset_index()

    # 判定 Label (最高分 >= 9)
    stats['label'] = stats['score_max'].apply(lambda x: 1 if x >= 9 else 0)

    # 透视成宽表
    wide = stats.pivot(index=['province', 'city'], columns='holiday', values=['n_records', 'score_mean', 'label'])
    
    # 展平列名：元旦_n_records, 春节_n_records...
    wide.columns = [f"{col[1]}_{col[0]}" for col in wide.columns]
    wide = wide.reset_index()

    # --- 7. 严格格式排序 ---
    target_cols = ['city', '元旦_n_records', '春节_n_records', '元旦_score_mean', '春节_score_mean', '元旦_label', '春节_label']
    # 补齐可能缺失的列（如果某些地方没抓到数据）
    for col in target_cols:
        if col not in wide.columns: wide[col] = 0
    
    final_df = wide[target_cols].copy()
    summary_path = os.path.join(out_dir, "national_summary_wide.xlsx")
    final_df.to_excel(summary_path, index=False)

    auto_resize_column(raw_path)
    auto_resize_column(summary_path)
    
    print(f"\n任务圆满完成！")
    print(f"1. 原始表: {raw_path}")
    print(f"2. 最终宽表: {summary_path}")

# --- 启动 ---
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--province", default="全国")
    args, _ = parser.parse_known_args() # 兼容 Jupyter

    target_list = list(PROVINCE_CITY_MAP.keys()) if args.province == "全国" else [args.province]
    run_national_task(target_list)


启动任务：共 362 个城市。直辖市深度：15


全城监测中:   1%|          | 2/362 [08:24<24:00:25, 240.07s/it]

跳过 天津-元旦: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 天津-春节: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 上海-元旦: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 上海-春节: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 重庆-元旦: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 重庆-春节: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 石家庄-元旦: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 石家庄-春节: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 唐山-元旦: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 唐山-春节: 
与页面的连接已断开。
版本: 4.1.1.2


全城监测中: 100%|██████████| 362/362 [19:02<00:00,  3.16s/it]  

跳过 秦皇岛-元旦: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 秦皇岛-春节: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 邯郸-元旦: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 邯郸-春节: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 邢台-元旦: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 邢台-春节: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 保定-元旦: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 保定-春节: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 张家口-元旦: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 张家口-春节: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 承德-元旦: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 承德-春节: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 沧州-元旦: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 沧州-春节: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 廊坊-元旦: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 廊坊-春节: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 衡水-元旦: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 衡水-春节: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 太原-元旦: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 太原-春节: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 大同-元旦: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 大同-春节: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 阳泉-元旦: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 阳泉-春节: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 长治-元旦: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 长治-春节: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 晋城-元旦: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 晋城-春节: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 朔州-元旦: 
与页面的连接已断开。
版本: 4.1.1.2
跳过 朔州-春节: 


任务圆满完成！
1. 原始表: xhs_result_20260311_210015\national_raw.xlsx
2. 最终宽表: xhs_result_20260311_210015\national_summary_wide.xlsx


: 

In [9]:
"""
无需任何第三方 Excel 库（不依赖 openpyxl、xlrd、calamine、pandas），
直接使用 Python 内置的 zipfile + xml.etree 解析 .xlsx 并计算审稿人要求的验证指标。
"""

import os
import zipfile
import xml.etree.ElementTree as ET

FILE_PATH = r"C:\Users\邓梦毫\Desktop\小红书无人机NLP\2025合并.xlsx"

def read_xlsx_native(xlsx_path):
    """
    使用 Python 标准库直接解压并解析 .xlsx 文件的 XML 数据
    """
    if not os.path.exists(xlsx_path):
        raise FileNotFoundError(f"未找到文件: {xlsx_path}")

    with zipfile.ZipFile(xlsx_path, 'r') as z:
        # 1. 读取共享字符串表 (Shared Strings)
        shared_strings = []
        if 'xl/sharedStrings.xml' in z.namelist():
            tree = ET.fromstring(z.read('xl/sharedStrings.xml'))
            ns = {'main': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
            for si in tree.findall('main:si', ns):
                t_elem = si.find('main:t', ns)
                if t_elem is not None and t_elem.text is not None:
                    shared_strings.append(t_elem.text)
                else:
                    text_parts = [t.text for t in si.findall('.//main:t', ns) if t.text]
                    shared_strings.append(''.join(text_parts))

        # 2. 读取第一个工作表
        sheet_path = 'xl/worksheets/sheet1.xml'
        tree = ET.fromstring(z.read(sheet_path))
        ns = {'main': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}

        rows_data = []
        for row_elem in tree.findall('.//main:row', ns):
            row_vals = {}
            for c_elem in row_elem.findall('main:c', ns):
                cell_ref = c_elem.get('r', '')
                col_letters = ''.join([ch for ch in cell_ref if ch.isalpha()])
                t_attr = c_elem.get('t', '')
                v_elem = c_elem.find('main:v', ns)
                
                val = ''
                if v_elem is not None and v_elem.text is not None:
                    raw_val = v_elem.text
                    if t_attr == 's':
                        idx = int(raw_val)
                        val = shared_strings[idx] if idx < len(shared_strings) else ''
                    else:
                        val = raw_val
                
                # 处理内联字符串 <is><t>
                if not val and c_elem.find('main:is/main:t', ns) is not None:
                    val = c_elem.find('main:is/main:t', ns).text or ''
                
                row_vals[col_letters] = val

            if row_vals:
                rows_data.append(row_vals)

    # 3. 转换为标准表格（以第一行为表头）
    if not rows_data:
        return []

    # 获取所有列字母并排序
    all_cols = []
    for r in rows_data:
        for c in r.keys():
            if c not in all_cols:
                all_cols.append(c)
    
    def col_to_idx(col_str):
        exp = 0
        idx = 0
        for char in reversed(col_str):
            idx += (ord(char) - ord('A') + 1) * (26 ** exp)
            exp += 1
        return idx
    
    sorted_cols = sorted(all_cols, key=col_to_idx)
    
    # 构建表头映射
    header_row = rows_data[0]
    col_to_header = {col: str(header_row.get(col, '')).strip() for col in sorted_cols}

    records = []
    for r in rows_data[1:]:
        rec = {}
        for col in sorted_cols:
            header_name = col_to_header.get(col, col)
            if header_name:
                rec[header_name] = r.get(col, '').strip()
        records.append(rec)

    return records


def parse_binary(val):
    """安全转换为 0 或 1"""
    if val is None:
        return None
    s = str(val).strip()
    if s == '':
        return None
    try:
        f = float(s)
        return int(f)
    except:
        return None


def calculate_metrics(y_true, y_pred, name=""):
    tp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and yp == 1)
    fp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 0 and yp == 1)
    fn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and yp == 0)
    tn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 0 and yp == 0)

    total = tp + fp + fn + tn
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy = (tp + tn) / total if total > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        'name': name,
        'N': total,
        'TP': tp,
        'FP': fp,
        'FN': fn,
        'TN': tn,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'Accuracy': accuracy,
        'Specificity': specificity
    }


def main():
    print("=" * 70)
    print("正在使用 Python 纯内置库（免除 openpyxl 依赖）读取 Excel 并评估...")
    print(f"目标文件: {FILE_PATH}")
    print("=" * 70)

    records = read_xlsx_native(FILE_PATH)
    print(f"成功解析 {len(records)} 行数据。\n")

    # 识别列名
    events = [
        ("元旦", "元旦_label", "元旦_true_label"),
        ("春节", "春节_label", "春节_true_label")
    ]

    all_yt = []
    all_yp = []

    for event_name, pred_col, true_col in events:
        y_true = []
        y_pred = []
        fp_cities = []
        fn_cities = []

        for r in records:
            city = r.get('city', r.get('城市', '未知城市'))
            pt = parse_binary(r.get(pred_col))
            tt = parse_binary(r.get(true_col))

            if pt is not None and tt is not None:
                y_pred.append(pt)
                y_true.append(tt)
                if pt == 1 and tt == 0:
                    fp_cities.append(city)
                elif pt == 0 and tt == 1:
                    fn_cities.append(city)

        m = calculate_metrics(y_true, y_pred, event_name)
        all_yt.extend(y_true)
        all_yp.extend(y_pred)

        print(f"▶ 【{event_name}】活动验证结果 (样本数 N = {m['N']}):")
        print(f"   - 混淆矩阵: TP={m['TP']}, FP={m['FP']}, FN={m['FN']}, TN={m['TN']}")
        print(f"   - 精确率 (Precision)   : {m['Precision']:.4f} ({m['Precision']*100:.2f}%)")
        print(f"   - 召回率 (Recall)      : {m['Recall']:.4f} ({m['Recall']*100:.2f}%)")
        print(f"   - F1 分数 (F1-Score)   : {m['F1']:.4f}")
        print(f"   - 特异度 (Specificity) : {m['Specificity']:.4f} ({m['Specificity']*100:.2f}%)")
        print(f"   - 准确率 (Accuracy)    : {m['Accuracy']:.4f} ({m['Accuracy']*100:.2f}%)")
        if fp_cities:
            print(f"   - 误报城市 (FP - 预测有实际无): {', '.join(fp_cities)}")
        if fn_cities:
            print(f"   - 漏报城市 (FN - 预测无实际有): {', '.join(fn_cities)}")
        print("-" * 70)

    # 总体指标
    om = calculate_metrics(all_yt, all_yp, "总体合并 (Overall Pooled)")
    print(f"▶ 【总体合并 (Overall Pooled)】(总样本数 N = {om['N']}):")
    print(f"   - 混淆矩阵: TP={om['TP']}, FP={om['FP']}, FN={om['FN']}, TN={om['TN']}")
    print(f"   - Precision   : {om['Precision']:.4f} ({om['Precision']*100:.2f}%)")
    print(f"   - Recall      : {om['Recall']:.4f} ({om['Recall']*100:.2f}%)")
    print(f"   - F1-Score    : {om['F1']:.4f}")
    print(f"   - Specificity : {om['Specificity']:.4f} ({om['Specificity']*100:.2f}%)")
    print(f"   - Accuracy    : {om['Accuracy']:.4f} ({om['Accuracy']*100:.2f}%)")
    print("=" * 70)


if __name__ == "__main__":
    main()


正在使用 Python 纯内置库（免除 openpyxl 依赖）读取 Excel 并评估...
目标文件: C:\Users\邓梦毫\Desktop\小红书无人机NLP\2025合并.xlsx
成功解析 419 行数据。

▶ 【元旦】活动验证结果 (样本数 N = 418):
   - 混淆矩阵: TP=109, FP=21, FN=14, TN=274
   - 精确率 (Precision)   : 0.8385 (83.85%)
   - 召回率 (Recall)      : 0.8862 (88.62%)
   - F1 分数 (F1-Score)   : 0.8617
   - 特异度 (Specificity) : 0.9288 (92.88%)
   - 准确率 (Accuracy)    : 0.9163 (91.63%)
   - 误报城市 (FP - 预测有实际无): 巴彦淖尔市, 白山市, 乐山, 巴中, 攀枝花, 菏泽市, 吕梁市, 崇左, 玉林, 南阳市, 鹤壁市, 邵阳, 南平, 黔西南布依族苗族自治州, 朝阳市, 安康市, 铜川市, 双鸭山市, 中卫, 果洛藏族自治州, 海东
   - 漏报城市 (FN - 预测无实际有): 嘉定, 徐汇, 浦东新区, 静安, 大理, 昆明, 南开区, 济南市, 郑州市, 岳阳, 贵阳, 大连市, 银川, 乌鲁木齐
----------------------------------------------------------------------
▶ 【春节】活动验证结果 (样本数 N = 418):
   - 混淆矩阵: TP=70, FP=11, FN=25, TN=312
   - 精确率 (Precision)   : 0.8642 (86.42%)
   - 召回率 (Recall)      : 0.7368 (73.68%)
   - F1 分数 (F1-Score)   : 0.7955
   - 特异度 (Specificity) : 0.9659 (96.59%)
   - 准确率 (Accuracy)    : 0.9139 (91.39%)
   - 误报城市 (FP - 预测有实际无): 巴彦淖尔市, 东丽区, 泰安市, 衢州, 荆门, 黄石, 黔南布依族苗族自治州